<!--Información del curso-->
<img align="left" style="padding-right:10px;" src="figuras/banner_cd.png">

<center><h1 style="font-size:2em;color:#2467C0"> Información de las sabanas del INE recolecta por el movimiento #YOSOY132 en la elección presidencial 2012  </h1></center>



<br>
<table>
<col width="650">
<col width="550">
<tr>
<td><img src="figuras/contamos.png" align="middle" style="width:700px"/></td>
<td>
    

Durante las elecciones presidenciales del año 2012, el colectivo #YOSOY132 pidió a la ciudadanía enviar la foto de su sabana con los resultados de su casilla.

    
</td>
</tr>
</table>

<div class="alert alert-success">
    
**Ejericicio**
* Hacer equipos de 2 o 3 integrantes.
* Hacer un análisis de los datos del directorio SABANAS_ELECCION_2012 
* Presentar los resultados obtenidos (enviar la presentación al profesor antes de comenzar la clase, junto con el código)
* Responder a la pregunta ¿Son confiables los datos? (Justificar su respuesta)
* Entregables por estudiantes: Presentación y archivo .ipynb
</div>

In [1]:
import json

In [24]:
# Visualizador de campos de datos JSON
def visualizar_json(ruta:str):
    print("JSON fields")
    with open(ruta) as archivo:
        datos = json.load(archivo)
        visualizar_campos_json(datos, 0)

def visualizar_campos_json(datos:dict, indent:int):   
    indent+=1
    for campo in datos:
        print("\t"*indent+campo)
        if isinstance(datos[campo], list):
            visualizar_campos_json(datos[campo][0], indent)
        if isinstance(datos[campo], dict):
            visualizar_campos_json(datos[campo], indent)

#USO
visualizar_json('SABANAS_INE_ELECCION_2012/file_id-0.json')


JSON fields
	sabanas
		id
		proyecto
		fuente
		imagen
		imagen_original
		entidad
		municipio
		distrito
		seccion
		tipoCasilla
		sha
		idcasilla
		revision
		resultados
			pan
			pri
			prd
			verde
			pt
			mov
			nuevaAlianza
			pri_verde
			prd_pt_mov
			prd_pt
			prd_mov
			pt_mov
			noRegistrados
			nulos


In [52]:
# Valores posibles: campo --> conjunto de valores
valores = dict()
# Resultados: partido --> numero de votos presidenciales
resultados = dict()
casillas_por_entidad = dict()
votos_por_entidad = dict()

# Recolector de datos
def leer_datos(directorio:str, ruta:str):
    with open(directorio+ruta) as archivo:
        for linea in archivo:
            archivo_trozo = linea.rstrip()
            leer_por_trozo(directorio, archivo_trozo)

def leer_por_trozo(directorio:str, archivo:str):
    with open(directorio+archivo) as trozo:
        datos = json.load(trozo)
        procesar(datos['sabanas'])

# Aplicación: Lector de sabanas
def procesar(sabana:dict):
    for casilla in sabana:
        # Metodos para recolectar
        # 1. valores
        obtener_valores_en(valores, casilla, 'tipoCasilla')
        obtener_valores_en(valores, casilla, 'entidad')
        # 2. Contar casillas por
        contar_casilla_por(casillas_por_entidad, casilla['entidad'])
        # 3. Conteo de votos
        total = contar_resultados(resultados, casilla['resultados'])
        contar_votos_por(votos_por_entidad, casilla['entidad'], total)

def contar_resultados(res:dict, resultados_c:dict):
    total_votos = 0
    for coalicion in resultados_c:
        num_votos_tipo = int(resultados_c[coalicion])
        res[coalicion] = res.get(coalicion, 0) + num_votos_tipo
        total_votos += num_votos_tipo
    return total_votos

def contar_casilla_por(res:dict, patron:str):
    res[patron] = res.get(patron, 0) + 1

def contar_votos_por(res:dict, patron:str, votos:int):
    res[patron] = res.get(patron, 0) + votos

def obtener_valores_en(res:dict, casilla:dict, campo:str):
    res[campo] = res.get(campo, set())
    res[campo].add(casilla[campo])

#USO
leer_datos('SABANAS_INE_ELECCION_2012/', 'nombre_archivos.txt')
print(casillas_por_entidad)

resultados

{'9': 6343, '5': 315, '2': 1337, '21': 1730, '25': 257, '7': 315, '19': 561, '15': 3731, '32': 101, '27': 363, '30': 1379, '11': 268, '17': 625, '14': 1121, '20': 500, '16': 312, '26': 280, '23': 431, '29': 161, '13': 434, '31': 259, '24': 426, '22': 356, '12': 181, '6': 106, '8': 360, '28': 147, '3': 110, '18': 106, '1': 189, '10': 86, '4': 110}


{'pan': 2247447,
 'pri': 2030613,
 'prd': 2253197,
 'verde': 93164,
 'pt': 255192,
 'mov': 208921,
 'nuevaAlianza': 203491,
 'pri_verde': 621972,
 'prd_pt_mov': 787276,
 'prd_pt': 128348,
 'prd_mov': 42425,
 'pt_mov': 22251,
 'noRegistrados': 10632,
 'nulos': 163943}

In [ ]:
#Hay que cambiarlo por los nombres de los candidatos
coaliciones_candidatos = {
    "PRI": {"pri", "verde"},
    "PAN": {"pan"},
    "PRD": {"prd", "pt", "mov"},
    "NA": {"nuevaAlianza"}
}
totales_candidatos = {}

for candidatos in coaliciones_candidatos:
    totales_candidatos[candidatos] = 0

In [43]:
for clave,valor in resultados.items():
  if clave in {'noRegistrados','nulos'}:
    continue
  partidos = set(clave.split('_'))

  for candidato , grupo in coaliciones_candidatos.items():
    if partidos <= grupo:
      totales_candidatos[candidato] += valor
      
print(totales_candidatos)

{'PRI': 2745749, 'PAN': 2247447, 'PRD': 3697610, 'NA': 203491}


In [ ]:
# Visualizar en matplotlib